In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sahilhussain2410/gridlock-round-2-ps1/jan to may police violation_anonymized791b166.csv
/kaggle/input/datasets/tanmaytripathi7525/gridlock-round2-theme2/Astram event data_anonymized - Astram event data_anonymizedb40ac87.csv


In [2]:
df=pd.read_csv('/kaggle/input/datasets/sahilhussain2410/gridlock-round-2-ps1/jan to may police violation_anonymized791b166.csv')

# PHASE A

## Dataset Audit

In [3]:
from pathlib import Path
import json
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 150)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

OUTPUT_DIR = Path("/kaggle/working/parking_intelligence")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: /kaggle/working/parking_intelligence


In [4]:
df.shape

(298450, 24)

In [5]:
memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)

print("Rows:", len(df))
print("Columns:", df.shape[1])
print(f"Memory usage: {memory_mb:.2f} MB")

basic_audit = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "non_null_count": df.notna().sum().values,
    "null_count": df.isna().sum().values,
    "null_percent": (df.isna().mean().values * 100),
    "unique_count": df.nunique(dropna=True).values,
    "memory_mb": [
        df[col].memory_usage(deep=True) / (1024 ** 2)
        for col in df.columns
    ]
})

basic_audit["unique_percent_of_rows"] = (
    basic_audit["unique_count"] / len(df) * 100
)

display(
    basic_audit.sort_values(
        ["null_percent", "unique_count"],
        ascending=[False, False]
    ).reset_index(drop=True)
)

Rows: 298450
Columns: 24
Memory usage: 320.50 MB


,column,dtype,non_null_count,null_count,null_percent,unique_count,memory_mb,unique_percent_of_rows
0,description,float64,0,298450,100.0000,0,2.2771,0.0000
1,closed_datetime,float64,0,298450,100.0000,0,2.2771,0.0000
2,action_taken_timestamp,float64,0,298450,100.0000,0,2.2771,0.0000
3,data_sent_to_scita_timestamp,object,42161,256289,85.8733,42161,10.9530,14.1267
4,validation_timestamp,object,173196,125254,41.9682,170115,16.1920,56.9995
5,updated_vehicle_number,object,173196,125254,41.9682,143133,14.0084,47.9588
6,updated_vehicle_type,object,173196,125254,41.9682,22,13.1287,0.0074
7,validation_status,object,173196,125254,41.9682,5,13.2390,0.0017
8,center_code,float64,287190,11260,3.7728,52,2.2771,0.0174
9,location,object,295409,3041,1.0189,10942,40.6035,3.6663


In [6]:
display(df.head())
print("Column names:")
print(df.columns.tolist())

,id,latitude,longitude,location,vehicle_number,vehicle_type,description,violation_type,offence_code,created_datetime,closed_datetime,modified_datetime,device_id,created_by_id,center_code,police_station,data_sent_to_scita,junction_name,action_taken_timestamp,data_sent_to_scita_timestamp,updated_vehicle_number,updated_vehicle_type,validation_status,validation_timestamp
0,FKID000000,12.9256,77.6187,"18th Main Road, Block 2, Koramangala, Bengaluru, Karnataka. Pin-560068 (India)",FKN00GL0000,CAR,NaN,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00,NaN,2023-11-28 04:48:04.582978+00,FKDEV00000,FKUSR00000,9.0000,Madiwala,True,No Junction,NaN,NaN,FKN00GL0000,MAXI-CAB,approved,2023-11-30 03:08:24.818+00
1,FKID000001,12.9055,77.7008,"Sarjapura Main Road, The Grove, Janatha Colony, Doddakannelli, Bengaluru, Karnataka. Pin-560035 (India)",FKN00GL0001,CAR,NaN,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00,NaN,2023-11-24 23:00:24.115257+00,FKDEV00001,FKUSR00001,82.0000,Bellandur,False,No Junction,NaN,NaN,NaN,NaN,NaN,NaN
2,FKID000002,12.9254,77.6185,"Koramangala 2nd Block, Kormangala West, Bengaluru South City Corporation, Bengaluru, Bangalore South, Bengaluru Urba...",FKN00GL0002,CAR,NaN,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00,NaN,2023-11-28 04:47:02.33776+00,FKDEV00000,FKUSR00000,9.0000,Madiwala,True,No Junction,NaN,NaN,FKN00GL0002,MAXI-CAB,approved,2023-11-30 03:08:56.998+00
3,FKID000003,12.9565,77.5186,"6th Cross Road, Manasa Layout, Nagarbhavi, Bengaluru, Karnataka. Pin-560072 (India)",FKN00GL0003,SCOOTER,NaN,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00,NaN,2023-11-18 04:46:57.216868+00,FKDEV00002,FKUSR00002,26.0000,Byatarayanapura,True,No Junction,NaN,NaN,FKN00GL0003,SCOOTER,approved,2023-11-18 23:35:12.428+00
4,FKID000004,12.9778,77.5805,"Kalidasa Road, Gandhinagar, Nehru Nagar, Bengaluru Central City Corporation, Bengaluru, Bangalore North, Bengaluru U...",FKN00GL0004,TANKER,NaN,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00,NaN,2023-11-28 02:44:50.46737+00,FKDEV00003,FKUSR00003,3.0000,Upparpet,True,BTP044 - Sagar Theatre Junction,NaN,NaN,FKN00GL0004,TANKER,approved,2023-11-30 03:11:32.796+00


Column names:
['id', 'latitude', 'longitude', 'location', 'vehicle_number', 'vehicle_type', 'description', 'violation_type', 'offence_code', 'created_datetime', 'closed_datetime', 'modified_datetime', 'device_id', 'created_by_id', 'center_code', 'police_station', 'data_sent_to_scita', 'junction_name', 'action_taken_timestamp', 'data_sent_to_scita_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'validation_status', 'validation_timestamp']


In [7]:
# missingness
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": df.isna().mean() * 100,
    "non_missing_count": df.notna().sum(),
    "unique_count": df.nunique(dropna=True)
}).sort_values(
    ["missing_percent", "unique_count"],
    ascending=[False, False]
)

display(missing_summary)

,missing_count,missing_percent,non_missing_count,unique_count
description,298450,100.0000,0,0
closed_datetime,298450,100.0000,0,0
action_taken_timestamp,298450,100.0000,0,0
data_sent_to_scita_timestamp,256289,85.8733,42161,42161
validation_timestamp,125254,41.9682,173196,170115
updated_vehicle_number,125254,41.9682,173196,143133
updated_vehicle_type,125254,41.9682,173196,22
validation_status,125254,41.9682,173196,5
center_code,11260,3.7728,287190,52
location,3041,1.0189,295409,10942


In [8]:
# inspect datetime
datetime_columns = [
    "created_datetime",
    "modified_datetime",
    "closed_datetime",
    "action_taken_timestamp",
    "data_sent_to_scita_timestamp",
    "validation_timestamp"
]

datetime_records = []
parsed_datetimes = {}

for col in datetime_columns:
    if col not in df.columns:
        continue

    parsed = pd.to_datetime(
        df[col],
        errors="coerce",
        utc=True
    )

    parsed_datetimes[col] = parsed

    valid = parsed.dropna()

    datetime_records.append({
        "column": col,
        "raw_non_null": int(df[col].notna().sum()),
        "successfully_parsed": int(parsed.notna().sum()),
        "failed_to_parse": int(
            df[col].notna().sum() - parsed.notna().sum()
        ),
        "minimum_utc": valid.min() if len(valid) else pd.NaT,
        "maximum_utc": valid.max() if len(valid) else pd.NaT,
        "unique_timestamps": int(valid.nunique())
    })

datetime_audit = pd.DataFrame(datetime_records)

display(datetime_audit)

,column,raw_non_null,successfully_parsed,failed_to_parse,minimum_utc,maximum_utc,unique_timestamps
0,created_datetime,298450,298445,5,2023-11-09 19:11:46+00:00,2024-04-08 17:30:46+00:00,94412
1,modified_datetime,298450,298450,0,2023-11-09 20:38:01.870979+00:00,2024-04-19 21:46:25.637149+00:00,298450
2,closed_datetime,0,0,0,NaT,NaT,0
3,action_taken_timestamp,0,0,0,NaT,NaT,0
4,data_sent_to_scita_timestamp,42161,42161,0,2024-03-22 10:54:47.389025+00:00,2024-04-19 02:34:46.290473+00:00,42161
5,validation_timestamp,173196,173030,166,2023-11-14 19:07:37.925000+00:00,2024-04-17 09:02:20.621000+00:00,169950


In [9]:
# main event period
created_utc = parsed_datetimes["created_datetime"]

valid_created = created_utc.dropna()

if len(valid_created):
    created_ist = valid_created.dt.tz_convert("Asia/Kolkata")

    print("Event period in UTC:")
    print("Start:", valid_created.min())
    print("End:  ", valid_created.max())

    print("\nEvent period in Asia/Kolkata:")
    print("Start:", created_ist.min())
    print("End:  ", created_ist.max())

    print("\nNumber of calendar days covered:")
    print((created_ist.max() - created_ist.min()).days + 1)
else:
    print("No valid created_datetime values were found.")

Event period in UTC:
Start: 2023-11-09 19:11:46+00:00
End:   2024-04-08 17:30:46+00:00

Event period in Asia/Kolkata:
Start: 2023-11-10 00:41:46+05:30
End:   2024-04-08 23:00:46+05:30

Number of calendar days covered:
151


In [10]:
# coordinate quality audit
latitude = pd.to_numeric(df["latitude"], errors="coerce")
longitude = pd.to_numeric(df["longitude"], errors="coerce")

coordinate_valid = (
    latitude.between(-90, 90)
    & longitude.between(-180, 180)
)

coordinate_complete = latitude.notna() & longitude.notna()

coordinate_summary = pd.DataFrame({
    "metric": [
        "total_rows",
        "complete_coordinate_rows",
        "missing_coordinate_rows",
        "valid_global_coordinate_rows",
        "invalid_global_coordinate_rows",
        "zero_zero_coordinates",
        "unique_latitudes",
        "unique_longitudes",
        "unique_coordinate_pairs"
    ],
    "value": [
        len(df),
        int(coordinate_complete.sum()),
        int((~coordinate_complete).sum()),
        int(coordinate_valid.sum()),
        int((coordinate_complete & ~coordinate_valid).sum()),
        int(((latitude == 0) & (longitude == 0)).sum()),
        int(latitude.nunique(dropna=True)),
        int(longitude.nunique(dropna=True)),
        int(
            pd.DataFrame({
                "latitude": latitude,
                "longitude": longitude
            }).drop_duplicates().shape[0]
        )
    ]
})

display(coordinate_summary)

,metric,value
0,total_rows,298450
1,complete_coordinate_rows,298450
2,missing_coordinate_rows,0
3,valid_global_coordinate_rows,298450
4,invalid_global_coordinate_rows,0
5,zero_zero_coordinates,0
6,unique_latitudes,177982
7,unique_longitudes,177378
8,unique_coordinate_pairs,209129


In [11]:
# coordinate distributions
coordinate_quantiles = pd.DataFrame({
    "latitude": latitude.quantile(
        [0, 0.001, 0.01, 0.25, 0.50, 0.75, 0.99, 0.999, 1]
    ),
    "longitude": longitude.quantile(
        [0, 0.001, 0.01, 0.25, 0.50, 0.75, 0.99, 0.999, 1]
    )
})

display(coordinate_quantiles)
# most frequent
top_coordinates = (
    df.groupby(["latitude", "longitude"], dropna=False)
    .size()
    .rename("record_count")
    .sort_values(ascending=False)
    .head(20)
    .reset_index()
)

display(top_coordinates)

,latitude,longitude
0.0000,12.8027,77.4426
0.0010,12.8230,77.4803
0.0100,12.8436,77.5115
0.2500,12.9633,77.5712
0.5000,12.9773,77.5841
0.7500,12.9975,77.6215
0.9900,13.1858,77.7200
0.9990,13.1957,77.7573
1.0000,13.2937,77.7717


,latitude,longitude,record_count
0,12.9995,77.5496,119
1,12.8763,77.5965,97
2,12.9341,77.6898,82
3,12.9992,77.5486,80
4,12.9991,77.5496,67
5,12.9345,77.6904,57
6,12.9344,77.6901,54
7,12.9300,77.6852,53
8,12.9991,77.5484,53
9,12.9669,77.6581,49


In [12]:
# identifier audit
identifier_columns = [
    "id",
    "vehicle_number",
    "device_id",
    "created_by_id"
]

identifier_audit = []

for col in identifier_columns:
    if col not in df.columns:
        continue

    counts = df[col].value_counts(dropna=False)

    identifier_audit.append({
        "column": col,
        "non_null_count": int(df[col].notna().sum()),
        "unique_count": int(df[col].nunique(dropna=True)),
        "duplicated_non_null_rows": int(
            df.loc[df[col].notna(), col].duplicated(keep=False).sum()
        ),
        "maximum_records_for_one_value": int(counts.max())
    })

identifier_audit = pd.DataFrame(identifier_audit)

display(identifier_audit)

,column,non_null_count,unique_count,duplicated_non_null_rows,maximum_records_for_one_value
0,id,298450,298450,0,1
1,vehicle_number,298450,231890,102147,55
2,device_id,298450,3070,298095,4344
3,created_by_id,298445,2666,298201,4099


In [13]:
# exact duplicate count
exact_duplicate_rows = df.duplicated(keep=False)
duplicate_ids = df["id"].duplicated(keep=False)

duplicate_summary = pd.DataFrame({
    "metric": [
        "exact_duplicate_rows_involved",
        "exact_duplicate_rows_after_first",
        "duplicate_id_rows_involved",
        "duplicate_id_rows_after_first"
    ],
    "value": [
        int(exact_duplicate_rows.sum()),
        int(df.duplicated().sum()),
        int(duplicate_ids.sum()),
        int(df["id"].duplicated().sum())
    ]
})

display(duplicate_summary)

,metric,value
0,exact_duplicate_rows_involved,0
1,exact_duplicate_rows_after_first,0
2,duplicate_id_rows_involved,0
3,duplicate_id_rows_after_first,0


In [14]:
# repeated capture of same incident
duplicate_work = pd.DataFrame({
    "vehicle_number": df["vehicle_number"].astype("string"),
    "latitude": latitude,
    "longitude": longitude,
    "created_utc": created_utc
})

duplicate_work["lat_5"] = duplicate_work["latitude"].round(5)
duplicate_work["lon_5"] = duplicate_work["longitude"].round(5)
duplicate_work["minute_bucket"] = duplicate_work["created_utc"].dt.floor("min")

duplicate_work["lat_4"] = duplicate_work["latitude"].round(4)
duplicate_work["lon_4"] = duplicate_work["longitude"].round(4)
duplicate_work["ten_minute_bucket"] = (
    duplicate_work["created_utc"].dt.floor("10min")
)

strict_columns = [
    "vehicle_number",
    "lat_5",
    "lon_5",
    "minute_bucket"
]

relaxed_columns = [
    "vehicle_number",
    "lat_4",
    "lon_4",
    "ten_minute_bucket"
]

strict_valid = duplicate_work[strict_columns].notna().all(axis=1)
relaxed_valid = duplicate_work[relaxed_columns].notna().all(axis=1)

strict_duplicate_mask = pd.Series(False, index=df.index)
relaxed_duplicate_mask = pd.Series(False, index=df.index)

strict_duplicate_mask.loc[strict_valid] = (
    duplicate_work.loc[strict_valid]
    .duplicated(strict_columns, keep=False)
)

relaxed_duplicate_mask.loc[relaxed_valid] = (
    duplicate_work.loc[relaxed_valid]
    .duplicated(relaxed_columns, keep=False)
)

strict_group_count = (
    duplicate_work.loc[strict_valid]
    .groupby(strict_columns)
    .size()
    .gt(1)
    .sum()
)

relaxed_group_count = (
    duplicate_work.loc[relaxed_valid]
    .groupby(relaxed_columns)
    .size()
    .gt(1)
    .sum()
)

near_duplicate_summary = pd.DataFrame({
    "candidate_rule": [
        "Strict: same vehicle + rounded 5-decimal coordinates + same minute",
        "Relaxed: same vehicle + rounded 4-decimal coordinates + same 10-minute window"
    ],
    "rows_involved": [
        int(strict_duplicate_mask.sum()),
        int(relaxed_duplicate_mask.sum())
    ],
    "percent_of_dataset": [
        strict_duplicate_mask.mean() * 100,
        relaxed_duplicate_mask.mean() * 100
    ],
    "duplicate_groups": [
        int(strict_group_count),
        int(relaxed_group_count)
    ]
})

display(near_duplicate_summary)

strict_candidate_sample = df.loc[
    strict_duplicate_mask,
    [
        "id",
        "vehicle_number",
        "latitude",
        "longitude",
        "created_datetime",
        "violation_type",
        "device_id",
        "created_by_id",
        "validation_status"
    ]
].copy()

strict_candidate_sample["created_utc"] = created_utc.loc[
    strict_candidate_sample.index
]

strict_candidate_sample = strict_candidate_sample.sort_values(
    ["vehicle_number", "created_utc", "latitude", "longitude"]
)

display(strict_candidate_sample.head(10))

,candidate_rule,rows_involved,percent_of_dataset,duplicate_groups
0,Strict: same vehicle + rounded 5-decimal coordinates + same minute,8666,2.9037,3288
1,Relaxed: same vehicle + rounded 4-decimal coordinates + same 10-minute window,9402,3.1503,3646


,id,vehicle_number,latitude,longitude,created_datetime,violation_type,device_id,created_by_id,validation_status,created_utc
47,FKID000047,FKN00GL0047,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,NaN,2024-03-10 07:10:46+00:00
48,FKID000048,FKN00GL0047,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,NaN,2024-03-10 07:10:46+00:00
49,FKID000049,FKN00GL0047,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,NaN,2024-03-10 07:10:46+00:00
101323,FKID101323,FKN00GL0047,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,approved,2024-03-10 07:10:46+00:00
101324,FKID101324,FKN00GL0047,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,NaN,2024-03-10 07:10:46+00:00
50,FKID000050,FKN00GL0048,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,NaN,2024-03-10 07:10:46+00:00
51,FKID000051,FKN00GL0048,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,NaN,2024-03-10 07:10:46+00:00
52,FKID000052,FKN00GL0048,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,NaN,2024-03-10 07:10:46+00:00
53,FKID000053,FKN00GL0048,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,NaN,2024-03-10 07:10:46+00:00
277732,FKID277732,FKN00GL0048,12.8419,77.6453,2024-03-10 07:10:46+00,"[""WRONG PARKING""]",FKDEV00028,FKUSR00028,approved,2024-03-10 07:10:46+00:00


In [15]:
# categorical columns check
categorical_columns = [
    "vehicle_type",
    "violation_type",
    "offence_code",
    "police_station",
    "junction_name",
    "validation_status",
    "data_sent_to_scita",
    "updated_vehicle_type"
]

categorical_audit = []

for col in categorical_columns:
    if col not in df.columns:
        continue

    categorical_audit.append({
        "column": col,
        "non_null_count": int(df[col].notna().sum()),
        "missing_percent": float(df[col].isna().mean() * 100),
        "unique_count": int(df[col].nunique(dropna=True)),
        "most_common_value": (
            df[col].value_counts(dropna=True).index[0]
            if df[col].notna().any()
            else None
        ),
        "most_common_count": (
            int(df[col].value_counts(dropna=True).iloc[0])
            if df[col].notna().any()
            else 0
        )
    })

categorical_audit = pd.DataFrame(categorical_audit)

display(categorical_audit)

,column,non_null_count,missing_percent,unique_count,most_common_value,most_common_count
0,vehicle_type,298450,0.0000,22,SCOOTER,94856
1,violation_type,298450,0.0000,991,"[""WRONG PARKING""]",138764
2,offence_code,298450,0.0000,991,[112],138764
3,police_station,298445,0.0017,54,Upparpet,34468
4,junction_name,298445,0.0017,169,No Junction,147880
5,validation_status,173196,41.9682,5,approved,115400
6,data_sent_to_scita,298450,0.0000,2,True,255893
7,updated_vehicle_type,173196,41.9682,22,SCOOTER,54867


In [16]:
# top category values
def show_top_values(data, column, n=15):
    print("=" * 100)
    print(f"TOP {n} VALUES: {column}")
    print("=" * 100)

    result = (
        data[column]
        .value_counts(dropna=False)
        .head(n)
        .rename_axis(column)
        .reset_index(name="count")
    )

    result["percent"] = result["count"] / len(data) * 100
    display(result)


for column in [
    "vehicle_type",
    "violation_type",
    "offence_code",
    "police_station",
    "junction_name",
    "validation_status"
]:
    show_top_values(df, column, n=15)

TOP 15 VALUES: vehicle_type


,vehicle_type,count,percent
0,SCOOTER,94856,31.7829
1,CAR,88870,29.7772
2,MOTOR CYCLE,40811,13.6743
3,PASSENGER AUTO,37813,12.6698
4,MAXI-CAB,11372,3.8104
5,LGV,8255,2.7660
6,GOODS AUTO,2934,0.9831
7,MOPED,2199,0.7368
8,PRIVATE BUS,1633,0.5472
9,VAN,1466,0.4912


TOP 15 VALUES: violation_type


,violation_type,count,percent
0,"[""WRONG PARKING""]",138764,46.4949
1,"[""NO PARKING""]",119576,40.0657
2,"[""PARKING IN A MAIN ROAD"",""WRONG PARKING""]",9472,3.1737
3,"[""PARKING IN A MAIN ROAD"",""NO PARKING""]",4818,1.6143
4,"[""WRONG PARKING"",""DEFECTIVE NUMBER PLATE""]",3317,1.1114
5,"[""NO PARKING"",""PARKING IN A MAIN ROAD""]",2449,0.8206
6,"[""NO PARKING"",""DEFECTIVE NUMBER PLATE""]",2380,0.7975
7,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]",1955,0.6551
8,"[""PARKING ON FOOTPATH"",""WRONG PARKING""]",1190,0.3987
9,"[""NO PARKING"",""WRONG PARKING""]",891,0.2985


TOP 15 VALUES: offence_code


...

,police_station,count,percent
0,Upparpet,34468,11.5490
1,Shivajinagar,28044,9.3965
2,Malleshwaram,22200,7.4384
3,HAL Old Airport,20819,6.9757
4,City Market,17646,5.9125
5,Vijayanagara,14652,4.9094
6,Rajajinagar,10998,3.6850
7,Kodigehalli,10916,3.6576
8,Magadi Road,8558,2.8675
9,Jeevanbheemanagar,6736,2.2570


TOP 15 VALUES: junction_name


,junction_name,count,percent
0,No Junction,147880,49.5493
1,BTP051 - Safina Plaza Junction,15449,5.1764
2,BTP082 - KR Market Junction,11538,3.8660
3,BTP040 - Elite Junction,10718,3.5912
4,BTP044 - Sagar Theatre Junction,10549,3.5346
5,BTP211 - Central Street Junction,5388,1.8053
6,BTP058 - Subbanna Junction,5189,1.7386
7,BTP027 - Modi Bridge Junction,4584,1.5359
8,BTP020 - Hosahalli Metro Station,4101,1.3741
9,BTP057 - Anand Rao Junction,3935,1.3185


TOP 15 VALUES: validation_status


,validation_status,count,percent
0,NaN,125254,41.9682
1,approved,115400,38.6664
2,rejected,49754,16.6708
3,created1,7044,2.3602
4,processing,678,0.2272
5,duplicate,320,0.1072


In [17]:
def concentration_report(data, column):
    counts = data[column].value_counts(dropna=True)

    total = counts.sum()

    if total == 0:
        return {
            "column": column,
            "unique_values": 0,
            "top_1_share_percent": np.nan,
            "top_10_share_percent": np.nan,
            "top_50_share_percent": np.nan,
            "median_records_per_value": np.nan,
            "maximum_records_per_value": np.nan
        }

    return {
        "column": column,
        "unique_values": int(len(counts)),
        "top_1_share_percent": counts.head(1).sum() / total * 100,
        "top_10_share_percent": counts.head(10).sum() / total * 100,
        "top_50_share_percent": counts.head(50).sum() / total * 100,
        "median_records_per_value": float(counts.median()),
        "maximum_records_per_value": int(counts.max())
    }


reporting_concentration = pd.DataFrame([
    concentration_report(df, "device_id"),
    concentration_report(df, "created_by_id"),
    concentration_report(df, "police_station"),
    concentration_report(df, "center_code")
])

display(reporting_concentration)
show_top_values(df, "device_id", n=15)
show_top_values(df, "created_by_id", n=15)

,column,unique_values,top_1_share_percent,top_10_share_percent,top_50_share_percent,median_records_per_value,maximum_records_per_value
0,device_id,3070,1.4555,7.5969,24.4215,16.0000,4344
1,created_by_id,2666,1.3735,7.9924,24.7798,21.0000,4099
2,police_station,54,11.5492,58.6497,99.1905,"3,293.5000",34468
3,center_code,52,12.0018,59.4265,99.5651,"3,293.5000",34468


TOP 15 VALUES: device_id


,device_id,count,percent
0,FKDEV00021,4344,1.4555
1,FKDEV00082,2268,0.7599
2,FKDEV00077,2156,0.7224
3,FKDEV00023,2145,0.7187
4,FKDEV00075,2095,0.7020
5,FKDEV00112,2012,0.6741
6,FKDEV00121,1991,0.6671
7,FKDEV00037,1960,0.6567
8,FKDEV00333,1879,0.6296
9,FKDEV00138,1823,0.6108


TOP 15 VALUES: created_by_id


,created_by_id,count,percent
0,FKUSR00021,4099,1.3734
1,FKUSR00332,3467,1.1617
2,FKUSR00082,2268,0.7599
3,FKUSR00077,2175,0.7288
4,FKUSR00023,2145,0.7187
5,FKUSR00075,2095,0.7020
6,FKUSR00113,2004,0.6715
7,FKUSR00122,1991,0.6671
8,FKUSR00115,1832,0.6138
9,FKUSR00068,1777,0.5954


In [18]:
# flag suspicious columns
column_flags = []

for col in df.columns:
    missing_percent = df[col].isna().mean() * 100
    unique_count = df[col].nunique(dropna=True)
    unique_ratio = unique_count / len(df)

    flags = []

    if missing_percent == 100:
        flags.append("completely_missing")

    if unique_count <= 1:
        flags.append("constant_or_empty")

    if missing_percent >= 40:
        flags.append("high_missingness")

    if unique_ratio >= 0.95:
        flags.append("near_unique_identifier")

    if col in {
        "modified_datetime",
        "validation_status",
        "validation_timestamp",
        "data_sent_to_scita",
        "data_sent_to_scita_timestamp",
        "updated_vehicle_number",
        "updated_vehicle_type"
    }:
        flags.append("possible_post_event_or_leakage")

    if col in {
        "vehicle_number",
        "updated_vehicle_number",
        "device_id",
        "created_by_id"
    }:
        flags.append("privacy_or_reporting_bias_risk")

    column_flags.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "missing_percent": missing_percent,
        "unique_count": unique_count,
        "unique_ratio": unique_ratio,
        "flags": ", ".join(flags) if flags else "none"
    })

column_flags = pd.DataFrame(column_flags)

display(
    column_flags.sort_values(
        ["flags", "missing_percent"],
        ascending=[True, False]
    ).reset_index(drop=True)
)

,column,dtype,missing_percent,unique_count,unique_ratio,flags
0,description,float64,100.0000,0,0.0000,"completely_missing, constant_or_empty, high_missingness"
1,closed_datetime,float64,100.0000,0,0.0000,"completely_missing, constant_or_empty, high_missingness"
2,action_taken_timestamp,float64,100.0000,0,0.0000,"completely_missing, constant_or_empty, high_missingness"
3,data_sent_to_scita_timestamp,object,85.8733,42161,0.1413,"high_missingness, possible_post_event_or_leakage"
4,updated_vehicle_type,object,41.9682,22,0.0001,"high_missingness, possible_post_event_or_leakage"
5,validation_status,object,41.9682,5,0.0000,"high_missingness, possible_post_event_or_leakage"
6,validation_timestamp,object,41.9682,170115,0.5700,"high_missingness, possible_post_event_or_leakage"
7,updated_vehicle_number,object,41.9682,143133,0.4796,"high_missingness, possible_post_event_or_leakage, privacy_or_reporting_bias_risk"
8,id,object,0.0000,298450,1.0000,near_unique_identifier
9,modified_datetime,object,0.0000,298450,1.0000,"near_unique_identifier, possible_post_event_or_leakage"


In [19]:
# saving audit reports
basic_audit.to_csv(
    OUTPUT_DIR / "basic_column_audit.csv",
    index=False
)

missing_summary.to_csv(
    OUTPUT_DIR / "missingness_summary.csv"
)

datetime_audit.to_csv(
    OUTPUT_DIR / "datetime_audit.csv",
    index=False
)

coordinate_summary.to_csv(
    OUTPUT_DIR / "coordinate_summary.csv",
    index=False
)

coordinate_quantiles.to_csv(
    OUTPUT_DIR / "coordinate_quantiles.csv"
)

duplicate_summary.to_csv(
    OUTPUT_DIR / "exact_duplicate_summary.csv",
    index=False
)

near_duplicate_summary.to_csv(
    OUTPUT_DIR / "near_duplicate_summary.csv",
    index=False
)

categorical_audit.to_csv(
    OUTPUT_DIR / "categorical_audit.csv",
    index=False
)

reporting_concentration.to_csv(
    OUTPUT_DIR / "reporting_concentration.csv",
    index=False
)

column_flags.to_csv(
    OUTPUT_DIR / "column_flags.csv",
    index=False
)

print("Audit reports saved to:")
print(OUTPUT_DIR)

Audit reports saved to:
/kaggle/working/parking_intelligence


In [20]:
# summary
print("=" * 80)
print("PHASE 1 — RESPONSE 1 SUMMARY")
print("=" * 80)

print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {memory_mb:.2f} MB")

print("\nCreated datetime coverage:")
display(
    datetime_audit.loc[
        datetime_audit["column"] == "created_datetime"
    ]
)

print("\nCoordinate summary:")
display(coordinate_summary)

print("\nExact duplicate summary:")
display(duplicate_summary)

print("\nNear-duplicate candidate summary:")
display(near_duplicate_summary)

print("\nReporting concentration:")
display(reporting_concentration)

print("\nColumn flags:")
display(
    column_flags.loc[
        column_flags["flags"] != "none",
        [
            "column",
            "missing_percent",
            "unique_count",
            "flags"
        ]
    ].reset_index(drop=True)
)

PHASE 1 — RESPONSE 1 SUMMARY
Dataset shape: (298450, 24)
Memory usage: 320.50 MB

Created datetime coverage:


,column,raw_non_null,successfully_parsed,failed_to_parse,minimum_utc,maximum_utc,unique_timestamps
0,created_datetime,298450,298445,5,2023-11-09 19:11:46+00:00,2024-04-08 17:30:46+00:00,94412



Coordinate summary:


,metric,value
0,total_rows,298450
1,complete_coordinate_rows,298450
2,missing_coordinate_rows,0
3,valid_global_coordinate_rows,298450
4,invalid_global_coordinate_rows,0
5,zero_zero_coordinates,0
6,unique_latitudes,177982
7,unique_longitudes,177378
8,unique_coordinate_pairs,209129



Exact duplicate summary:


...

,candidate_rule,rows_involved,percent_of_dataset,duplicate_groups
0,Strict: same vehicle + rounded 5-decimal coordinates + same minute,8666,2.9037,3288
1,Relaxed: same vehicle + rounded 4-decimal coordinates + same 10-minute window,9402,3.1503,3646



Reporting concentration:


,column,unique_values,top_1_share_percent,top_10_share_percent,top_50_share_percent,median_records_per_value,maximum_records_per_value
0,device_id,3070,1.4555,7.5969,24.4215,16.0000,4344
1,created_by_id,2666,1.3735,7.9924,24.7798,21.0000,4099
2,police_station,54,11.5492,58.6497,99.1905,"3,293.5000",34468
3,center_code,52,12.0018,59.4265,99.5651,"3,293.5000",34468



Column flags:


,column,missing_percent,unique_count,flags
0,id,0.0000,298450,near_unique_identifier
1,vehicle_number,0.0000,231890,privacy_or_reporting_bias_risk
2,description,100.0000,0,"completely_missing, constant_or_empty, high_missingness"
3,closed_datetime,100.0000,0,"completely_missing, constant_or_empty, high_missingness"
4,modified_datetime,0.0000,298450,"near_unique_identifier, possible_post_event_or_leakage"
5,device_id,0.0000,3070,privacy_or_reporting_bias_risk
6,created_by_id,0.0017,2666,privacy_or_reporting_bias_risk
7,data_sent_to_scita,0.0000,2,possible_post_event_or_leakage
8,action_taken_timestamp,100.0000,0,"completely_missing, constant_or_empty, high_missingness"
9,data_sent_to_scita_timestamp,85.8733,42161,"high_missingness, possible_post_event_or_leakage"


## Core Cleaning and Leakage Separation

In [21]:
events_clean=pd.read_csv('/kaggle/input/datasets/sahilhussain2410/gridlock-round-2-ps1/jan to may police violation_anonymized791b166.csv')
events_clean.insert(
    0,
    "source_row",
    np.arange(len(events_clean), dtype=np.int64)
)

print("Starting shape:", events_clean.shape)

Starting shape: (298450, 25)


In [22]:
# remove completely empty columns
empty_columns = [
    "description",
    "closed_datetime",
    "action_taken_timestamp"
]

for col in empty_columns:
    assert col in events_clean.columns
    assert events_clean[col].isna().all(), (
        f"{col} is no longer completely empty."
    )

events_clean = events_clean.drop(columns=empty_columns)

print("Removed:", empty_columns)
print("Shape after removal:", events_clean.shape)

Removed: ['description', 'closed_datetime', 'action_taken_timestamp']
Shape after removal: (298450, 22)


In [23]:
# standardize the missing string columns
string_columns = events_clean.select_dtypes(
    include=["object", "string"]
).columns

for col in string_columns:
    events_clean[col] = (
        events_clean[col]
        .astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "NaN": pd.NA,
            "None": pd.NA,
            "NULL": pd.NA
        })
    )

print("String columns standardized:", len(string_columns))

String columns standardized: 17


In [24]:
# standardize selected categorical fields
uppercase_columns = [
    "vehicle_number",
    "vehicle_type",
    "updated_vehicle_number",
    "updated_vehicle_type",
    "validation_status"
]

for col in uppercase_columns:
    if col in events_clean.columns:
        events_clean[col] = events_clean[col].str.upper()

# Normalize repeated internal whitespace in scalar categories.
scalar_category_columns = [
    "vehicle_type",
    "updated_vehicle_type",
    "police_station",
    "junction_name",
    "validation_status"
]

for col in scalar_category_columns:
    if col in events_clean.columns:
        events_clean[col] = (
            events_clean[col]
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

print("Categorical normalization completed.")

Categorical normalization completed.


In [25]:
# convert numeric columns
events_clean["latitude"] = pd.to_numeric(
    events_clean["latitude"],
    errors="coerce"
)

events_clean["longitude"] = pd.to_numeric(
    events_clean["longitude"],
    errors="coerce"
)

events_clean["center_code"] = pd.to_numeric(
    events_clean["center_code"],
    errors="coerce"
).astype("Int64")



In [26]:
def convert_to_nullable_boolean(series):
    if pd.api.types.is_bool_dtype(series):
        return series.astype("boolean")

    normalized = series.astype("string").str.strip().str.lower()

    mapping = {
        "true": True,
        "1": True,
        "yes": True,
        "false": False,
        "0": False,
        "no": False
    }

    return normalized.map(mapping).astype("boolean")


events_clean["data_sent_to_scita"] = convert_to_nullable_boolean(
    events_clean["data_sent_to_scita"]
)

print(events_clean["data_sent_to_scita"].value_counts(dropna=False))

data_sent_to_scita
True     255893
False     42557
Name: count, dtype: Int64


In [27]:
# parse the timestamps columns to ist
datetime_mapping = {
    "created_datetime": "created_at_utc",
    "modified_datetime": "modified_at_utc",
    "data_sent_to_scita_timestamp": "scita_sent_at_utc",
    "validation_timestamp": "validated_at_utc"
}

for raw_col, parsed_col in datetime_mapping.items():
    events_clean[parsed_col] = pd.to_datetime(
        events_clean[raw_col],
        errors="coerce",
        utc=True
    )

events_clean["created_at_ist"] = (
    events_clean["created_at_utc"]
    .dt.tz_convert("Asia/Kolkata")
)

events_clean["created_time_parse_failed"] = (
    events_clean["created_datetime"].notna()
    & events_clean["created_at_utc"].isna()
)

events_clean["validation_time_parse_failed"] = (
    events_clean["validation_timestamp"].notna()
    & events_clean["validated_at_utc"].isna()
)

print(
    "Invalid created timestamps:",
    events_clean["created_time_parse_failed"].sum()
)

print(
    "Invalid validation timestamps:",
    events_clean["validation_time_parse_failed"].sum()
)

Invalid created timestamps: 5
Invalid validation timestamps: 166


In [28]:
# isolate invalid event time rows
invalid_event_time_rows = events_clean.loc[
    events_clean["created_at_utc"].isna()
].copy()

events_clean = events_clean.loc[
    events_clean["created_at_utc"].notna()
].copy()

events_clean = events_clean.reset_index(drop=True)

print("Rows isolated:", len(invalid_event_time_rows))
print("Remaining rows:", len(events_clean))

Rows isolated: 5
Remaining rows: 298445


In [29]:
events_clean["vehicle_number_event"] = (
    events_clean["vehicle_number"]
)

events_clean["vehicle_type_event"] = (
    events_clean["vehicle_type"]
)

events_clean["vehicle_number_resolved_audit"] = (
    events_clean["updated_vehicle_number"]
    .combine_first(events_clean["vehicle_number"])
)

events_clean["vehicle_type_resolved_audit"] = (
    events_clean["updated_vehicle_type"]
    .combine_first(events_clean["vehicle_type"])
)

events_clean["vehicle_number_was_updated"] = (
    events_clean["updated_vehicle_number"].notna()
    & (
        events_clean["updated_vehicle_number"]
        != events_clean["vehicle_number"]
    )
)

events_clean["vehicle_type_was_updated"] = (
    events_clean["updated_vehicle_type"].notna()
    & (
        events_clean["updated_vehicle_type"]
        != events_clean["vehicle_type"]
    )
)

print(
    "Vehicle numbers changed after validation:",
    events_clean["vehicle_number_was_updated"].sum()
)

print(
    "Vehicle types changed after validation:",
    events_clean["vehicle_type_was_updated"].sum()
)

Vehicle numbers changed after validation: 1774
Vehicle types changed after validation: 6169


In [30]:
# privacy safe vehicle tokens
import hashlib

unique_vehicle_numbers = (
    events_clean["vehicle_number_event"]
    .dropna()
    .unique()
)

vehicle_token_map = {
    vehicle_number: hashlib.sha256(
        f"PARKGUARD_V1|{vehicle_number}".encode("utf-8")
    ).hexdigest()[:20]
    for vehicle_number in unique_vehicle_numbers
}

events_clean["vehicle_token"] = (
    events_clean["vehicle_number_event"]
    .map(vehicle_token_map)
    .astype("string")
)

print("Raw unique vehicles:", events_clean["vehicle_number_event"].nunique())
print("Unique vehicle tokens:", events_clean["vehicle_token"].nunique())

Raw unique vehicles: 231890
Unique vehicle tokens: 231890


In [31]:
# fill missing values
categorical_fill_values = {
    "location": "UNKNOWN LOCATION",
    "device_id": "UNKNOWN DEVICE",
    "created_by_id": "UNKNOWN CREATOR",
    "police_station": "UNKNOWN POLICE STATION",
    "junction_name": "UNKNOWN JUNCTION",
    "vehicle_type_event": "UNKNOWN VEHICLE TYPE"
}

for col, fill_value in categorical_fill_values.items():
    events_clean[col] = events_clean[col].fillna(fill_value)

print("Remaining missing values in selected categories:")

display(
    events_clean[
        list(categorical_fill_values.keys())
    ].isna().sum().to_frame("missing_count")
)

Remaining missing values in selected categories:


,missing_count
location,0
device_id,0
created_by_id,0
police_station,0
junction_name,0
vehicle_type_event,0


In [32]:
# defining columns roles
column_roles = {
    # Record tracking
    "source_row": "record_key",
    "id": "record_key",

    # Directly observed event-time information
    "latitude": "real_time_spatial",
    "longitude": "real_time_spatial",
    "created_at_utc": "real_time_temporal",
    "created_at_ist": "real_time_temporal",
    "vehicle_type_event": "real_time_behavioural",
    "vehicle_token": "aggregate_behaviour_only",
    "violation_type": "real_time_behavioural",
    "offence_code": "real_time_behavioural",
    "junction_name": "real_time_context",

    # Display/context fields
    "location": "display_context",

    # Observation mechanism: aggregate bias/confidence only
    "device_id": "observation_bias_only",
    "created_by_id": "observation_bias_only",
    "center_code": "observation_bias_only",
    "police_station": "observation_bias_only",

    # Original sensitive field: never direct model feature
    "vehicle_number": "sensitive_excluded",
    "vehicle_number_event": "sensitive_excluded",

    # Raw timestamp strings retained for traceability
    "created_datetime": "raw_traceability",
    "modified_datetime": "post_event_audit",

    # Post-event fields: prohibited as direct predictive inputs
    "modified_at_utc": "post_event_audit",
    "data_sent_to_scita": "post_event_audit",
    "data_sent_to_scita_timestamp": "post_event_audit",
    "scita_sent_at_utc": "post_event_audit",
    "updated_vehicle_number": "post_event_audit",
    "updated_vehicle_type": "post_event_audit",
    "validation_status": "post_event_audit",
    "validation_timestamp": "post_event_audit",
    "validated_at_utc": "post_event_audit",
    "vehicle_number_resolved_audit": "post_event_audit",
    "vehicle_type_resolved_audit": "post_event_audit",
    "vehicle_number_was_updated": "post_event_audit",
    "vehicle_type_was_updated": "post_event_audit",
    "validation_time_parse_failed": "post_event_audit",

    # Quality flag
    "created_time_parse_failed": "quality_flag"
}

column_role_table = pd.DataFrame({
    "column": events_clean.columns,
    "role": [
        column_roles.get(col, "unclassified")
        for col in events_clean.columns
    ]
})

display(column_role_table)

,column,role
0,source_row,record_key
1,id,record_key
2,latitude,real_time_spatial
3,longitude,real_time_spatial
4,location,display_context
5,vehicle_number,sensitive_excluded
6,vehicle_type,unclassified
7,violation_type,real_time_behavioural
8,offence_code,real_time_behavioural
9,created_datetime,raw_traceability


In [33]:
unclassified_columns = column_role_table.loc[
    column_role_table["role"] == "unclassified",
    "column"
].tolist()

print("Unclassified columns:", unclassified_columns)

Unclassified columns: ['vehicle_type']


In [34]:
# creating reusable role list
record_key_columns = column_role_table.loc[
    column_role_table["role"] == "record_key",
    "column"
].tolist()

real_time_columns = column_role_table.loc[
    column_role_table["role"].str.startswith("real_time"),
    "column"
].tolist()

observation_bias_columns = column_role_table.loc[
    column_role_table["role"] == "observation_bias_only",
    "column"
].tolist()

post_event_columns = column_role_table.loc[
    column_role_table["role"] == "post_event_audit",
    "column"
].tolist()

directly_excluded_columns = column_role_table.loc[
    column_role_table["role"].isin([
        "post_event_audit",
        "sensitive_excluded",
        "raw_traceability",
        "display_context"
    ]),
    "column"
].tolist()

print("Real-time columns:")
print(real_time_columns)

print("\nObservation-bias columns:")
print(observation_bias_columns)

print("\nPost-event columns:")
print(post_event_columns)

print("\nDirectly excluded from predictive modelling:")
print(directly_excluded_columns)

Real-time columns:
['latitude', 'longitude', 'violation_type', 'offence_code', 'junction_name', 'created_at_utc', 'created_at_ist', 'vehicle_type_event']

Observation-bias columns:
['device_id', 'created_by_id', 'center_code', 'police_station']

Post-event columns:
['modified_datetime', 'data_sent_to_scita', 'data_sent_to_scita_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'validation_status', 'validation_timestamp', 'modified_at_utc', 'scita_sent_at_utc', 'validated_at_utc', 'validation_time_parse_failed', 'vehicle_number_resolved_audit', 'vehicle_type_resolved_audit', 'vehicle_number_was_updated', 'vehicle_type_was_updated']

Directly excluded from predictive modelling:
['location', 'vehicle_number', 'created_datetime', 'modified_datetime', 'data_sent_to_scita', 'data_sent_to_scita_timestamp', 'updated_vehicle_number', 'updated_vehicle_type', 'validation_status', 'validation_timestamp', 'modified_at_utc', 'scita_sent_at_utc', 'validated_at_utc', 'validation_time_parse

In [35]:
# validating the cleaned dataset
assert len(events_clean) == len(df) - 5
assert events_clean["id"].is_unique
assert events_clean["created_at_utc"].notna().all()
assert events_clean["latitude"].notna().all()
assert events_clean["longitude"].notna().all()
assert events_clean["vehicle_token"].nunique() == (
    events_clean["vehicle_number_event"].nunique()
)

print("All cleaning assertions passed.")

All cleaning assertions passed.


In [36]:
# summary of cleaning
cleaning_summary = pd.DataFrame({
    "metric": [
        "raw_rows",
        "clean_rows",
        "rows_removed_due_to_invalid_event_time",
        "columns_removed_as_completely_empty",
        "invalid_validation_timestamps_retained_for_audit",
        "vehicle_numbers_changed_after_validation",
        "vehicle_types_changed_after_validation",
        "unique_vehicle_tokens"
    ],
    "value": [
        len(df),
        len(events_clean),
        len(invalid_event_time_rows),
        len(empty_columns),
        int(events_clean["validation_time_parse_failed"].sum()),
        int(events_clean["vehicle_number_was_updated"].sum()),
        int(events_clean["vehicle_type_was_updated"].sum()),
        int(events_clean["vehicle_token"].nunique())
    ]
})

display(cleaning_summary)

,metric,value
0,raw_rows,298450
1,clean_rows,298445
2,rows_removed_due_to_invalid_event_time,5
3,columns_removed_as_completely_empty,3
4,invalid_validation_timestamps_retained_for_audit,166
5,vehicle_numbers_changed_after_validation,1774
6,vehicle_types_changed_after_validation,6169
7,unique_vehicle_tokens,231890


In [37]:
# saving cleaned dataset
clean_data_path = OUTPUT_DIR / "events_clean_stage1.parquet"
invalid_rows_path = OUTPUT_DIR / "invalid_event_time_rows.csv"
roles_path = OUTPUT_DIR / "column_roles.csv"
summary_path = OUTPUT_DIR / "cleaning_summary.csv"

events_clean.to_parquet(
    clean_data_path,
    index=False
)

invalid_event_time_rows.to_csv(
    invalid_rows_path,
    index=False
)

column_role_table.to_csv(
    roles_path,
    index=False
)

cleaning_summary.to_csv(
    summary_path,
    index=False
)

print("Saved:")
print(clean_data_path)
print(invalid_rows_path)
print(roles_path)
print(summary_path)

Saved:
/kaggle/working/parking_intelligence/events_clean_stage1.parquet
/kaggle/working/parking_intelligence/invalid_event_time_rows.csv
/kaggle/working/parking_intelligence/column_roles.csv
/kaggle/working/parking_intelligence/cleaning_summary.csv


In [38]:
events_clean.shape

(298445, 36)

## Parse offences and Standardized vehicle categories

In [39]:
# 
column_roles["vehicle_type"] = "raw_traceability"

column_role_table = pd.DataFrame({
    "column": events_clean.columns,
    "role": [
        column_roles.get(col, "unclassified")
        for col in events_clean.columns
    ]
})

unclassified_columns = column_role_table.loc[
    column_role_table["role"] == "unclassified",
    "column"
].tolist()

print("Unclassified columns:", unclassified_columns)

Unclassified columns: []


In [40]:
import ast
import json
import re

from collections import Counter
from itertools import combinations
# function to convert python list and naormalize spacing in them
def normalize_violation_label(value):
    value = str(value).strip().upper()
    value = re.sub(r"\s+", " ", value)
    return value


def normalize_offence_code(value):
    value = str(value).strip()

    # Convert values such as "112.0" into "112".
    if re.fullmatch(r"\d+(\.0+)?", value):
        return str(int(float(value)))

    return re.sub(r"\s+", " ", value).upper()


def safe_parse_list(value, normalizer):
    if pd.isna(value):
        return [], True

    if isinstance(value, (list, tuple, set, np.ndarray)):
        parsed = list(value)

    else:
        text = str(value).strip()

        try:
            parsed = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            try:
                parsed = json.loads(text)
            except (json.JSONDecodeError, TypeError):
                return [], False

    if not isinstance(parsed, (list, tuple, set, np.ndarray)):
        parsed = [parsed]

    cleaned = []

    for item in parsed:
        if item is None or pd.isna(item):
            continue

        normalized = normalizer(item)

        if normalized:
            cleaned.append(normalized)

    return cleaned, True

In [41]:
# Parse violation labels and offence codes
violation_parse_result = events_clean["violation_type"].map(
    lambda value: safe_parse_list(
        value,
        normalize_violation_label
    )
)

code_parse_result = events_clean["offence_code"].map(
    lambda value: safe_parse_list(
        value,
        normalize_offence_code
    )
)

events_clean["violation_labels"] = violation_parse_result.map(
    lambda result: result[0]
)

events_clean["violation_parse_ok"] = violation_parse_result.map(
    lambda result: result[1]
)

events_clean["offence_codes"] = code_parse_result.map(
    lambda result: result[0]
)

events_clean["offence_code_parse_ok"] = code_parse_result.map(
    lambda result: result[1]
)

print(
    "Violation parse failures:",
    (~events_clean["violation_parse_ok"]).sum()
)

print(
    "Offence-code parse failures:",
    (~events_clean["offence_code_parse_ok"]).sum()
)

Violation parse failures: 0
Offence-code parse failures: 0


In [42]:
#removes repeated labels within one event and sorts combinations into a canonical form
def unique_preserving_order(values):
    return list(dict.fromkeys(values))


events_clean["violation_labels_unique"] = (
    events_clean["violation_labels"]
    .map(unique_preserving_order)
)

events_clean["offence_codes_unique"] = (
    events_clean["offence_codes"]
    .map(unique_preserving_order)
)

events_clean["violation_count_raw"] = (
    events_clean["violation_labels"].map(len)
)

events_clean["violation_count"] = (
    events_clean["violation_labels_unique"].map(len)
)

events_clean["offence_code_count"] = (
    events_clean["offence_codes_unique"].map(len)
)

events_clean["violation_combo_canonical"] = (
    events_clean["violation_labels_unique"]
    .map(
        lambda labels: " | ".join(sorted(labels))
        if labels
        else "UNKNOWN"
    )
)

events_clean["offence_combo_canonical"] = (
    events_clean["offence_codes_unique"]
    .map(
        lambda codes: " | ".join(sorted(codes))
        if codes
        else "UNKNOWN"
    )
)

print(
    "Raw violation combinations:",
    events_clean["violation_type"].nunique()
)

print(
    "Canonical violation combinations:",
    events_clean["violation_combo_canonical"].nunique()
)

Raw violation combinations: 991
Canonical violation combinations: 414


In [43]:
# every violation label should have a corresponding offence code. 
# This cell checks empty lists, parsing failures, duplicate labels and label-code count mismatches.
events_clean["label_code_count_match"] = (
    events_clean["violation_count"]
    == events_clean["offence_code_count"]
)

offence_parsing_audit = pd.DataFrame({
    "metric": [
        "total_events",
        "violation_parse_failures",
        "offence_code_parse_failures",
        "events_with_zero_labels",
        "events_with_zero_codes",
        "events_with_duplicate_labels_inside_list",
        "label_code_count_mismatches",
        "unique_raw_violation_combinations",
        "unique_canonical_violation_combinations",
        "unique_atomic_violation_labels",
        "unique_atomic_offence_codes"
    ],
    "value": [
        len(events_clean),
        int((~events_clean["violation_parse_ok"]).sum()),
        int((~events_clean["offence_code_parse_ok"]).sum()),
        int((events_clean["violation_count"] == 0).sum()),
        int((events_clean["offence_code_count"] == 0).sum()),
        int(
            (
                events_clean["violation_count_raw"]
                > events_clean["violation_count"]
            ).sum()
        ),
        int((~events_clean["label_code_count_match"]).sum()),
        int(events_clean["violation_type"].nunique()),
        int(events_clean["violation_combo_canonical"].nunique()),
        int(
            events_clean["violation_labels_unique"]
            .explode()
            .dropna()
            .nunique()
        ),
        int(
            events_clean["offence_codes_unique"]
            .explode()
            .dropna()
            .nunique()
        )
    ]
})

display(offence_parsing_audit)

,metric,value
0,total_events,298445
1,violation_parse_failures,0
2,offence_code_parse_failures,0
3,events_with_zero_labels,0
4,events_with_zero_codes,0
5,events_with_duplicate_labels_inside_list,0
6,label_code_count_mismatches,0
7,unique_raw_violation_combinations,991
8,unique_canonical_violation_combinations,414
9,unique_atomic_violation_labels,27


In [44]:
# inspacting mismatches
label_code_mismatches = events_clean.loc[
    ~events_clean["label_code_count_match"],
    [
        "id",
        "violation_type",
        "offence_code",
        "violation_labels_unique",
        "offence_codes_unique"
    ]
]

display(label_code_mismatches.head(20))

,id,violation_type,offence_code,violation_labels_unique,offence_codes_unique


In [45]:
# creates one row per atomic violation, making offence frequency and label-code analysis easier
label_long = (
    events_clean[
        ["source_row", "id", "violation_labels_unique"]
    ]
    .explode("violation_labels_unique")
    .rename(
        columns={
            "violation_labels_unique": "violation_label"
        }
    )
    .dropna(subset=["violation_label"])
)

label_long["offence_position"] = (
    label_long.groupby("source_row").cumcount()
)

#---------
code_long = (
    events_clean[
        ["source_row", "offence_codes_unique"]
    ]
    .explode("offence_codes_unique")
    .rename(
        columns={
            "offence_codes_unique": "offence_code_atomic"
        }
    )
    .dropna(subset=["offence_code_atomic"])
)

code_long["offence_position"] = (
    code_long.groupby("source_row").cumcount()
)

#--------------
event_offence_long = label_long.merge(
    code_long,
    on=["source_row", "offence_position"],
    how="outer",
    validate="one_to_one"
)

print("Rows in event-offence table:", len(event_offence_long))
display(event_offence_long.head(10))

Rows in event-offence table: 348449


,source_row,id,violation_label,offence_position,offence_code_atomic
0,0,FKID000000,WRONG PARKING,0,112
1,0,FKID000000,PARKING NEAR ROAD CROSSING,1,104
2,1,FKID000001,NO PARKING,0,113
3,2,FKID000002,WRONG PARKING,0,112
4,2,FKID000002,PARKING IN A MAIN ROAD,1,107
5,3,FKID000003,NO PARKING,0,113
6,4,FKID000004,NO PARKING,0,113
7,5,FKID000005,NO PARKING,0,113
8,6,FKID000006,NO PARKING,0,113
9,7,FKID000007,WRONG PARKING,0,112


In [46]:
# This measures actual individual offences rather than raw combinations. 
# The table also reports the number of unique events containing each offence.

atomic_offence_frequency = (
    event_offence_long
    .groupby("violation_label")
    .agg(
        occurrence_count=("source_row", "size"),
        unique_event_count=("source_row", "nunique"),
        unique_codes=("offence_code_atomic", "nunique")
    )
    .sort_values("unique_event_count", ascending=False)
    .reset_index()
)

atomic_offence_frequency["event_percent"] = (
    atomic_offence_frequency["unique_event_count"]
    / len(events_clean)
    * 100
)

display(atomic_offence_frequency.head(30))

,violation_label,occurrence_count,unique_event_count,unique_codes,event_percent
0,WRONG PARKING,164974,164974,1,55.2779
1,NO PARKING,139048,139048,1,46.5908
2,PARKING IN A MAIN ROAD,23943,23943,1,8.0226
3,DEFECTIVE NUMBER PLATE,7847,7847,1,2.6293
4,PARKING ON FOOTPATH,3757,3757,1,1.2589
5,PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC,2403,2403,1,0.8052
6,DOUBLE PARKING,2037,2037,1,0.6825
7,PARKING NEAR ROAD CROSSING,1687,1687,1,0.5653
8,REFUSE TO GO FOR HIRE,887,887,1,0.2972
9,PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS,525,525,1,0.1759


In [47]:
# A violation label should ideally map to one stable offence code. 
# Labels associated with multiple codes need investigation before model features are created.
label_code_mapping = (
    event_offence_long
    .dropna(
        subset=[
            "violation_label",
            "offence_code_atomic"
        ]
    )
    .groupby(
        ["violation_label", "offence_code_atomic"]
    )
    .size()
    .rename("pair_count")
    .reset_index()
)

label_code_consistency = (
    label_code_mapping
    .groupby("violation_label")
    .agg(
        number_of_codes=("offence_code_atomic", "nunique"),
        total_occurrences=("pair_count", "sum")
    )
    .sort_values(
        ["number_of_codes", "total_occurrences"],
        ascending=[False, False]
    )
    .reset_index()
)

display(label_code_consistency.head(30))

,violation_label,number_of_codes,total_occurrences
0,WRONG PARKING,1,164974
1,NO PARKING,1,139048
2,PARKING IN A MAIN ROAD,1,23943
3,DEFECTIVE NUMBER PLATE,1,7847
4,PARKING ON FOOTPATH,1,3757
5,PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC,1,2403
6,DOUBLE PARKING,1,2037
7,PARKING NEAR ROAD CROSSING,1,1687
8,REFUSE TO GO FOR HIRE,1,887
9,PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS,1,525


In [48]:
# inspect lables linked to multiple codes
multi_code_labels = label_code_consistency.loc[
    label_code_consistency["number_of_codes"] > 1
]

print(
    "Labels associated with multiple offence codes:",
    len(multi_code_labels)
)

display(multi_code_labels)

Labels associated with multiple offence codes: 0


,violation_label,number_of_codes,total_occurrences


In [49]:
# This finds offences that frequently appear together in the same event.
# It can later support offence-profile features and semantic obstruction modelling.
offence_pair_counter = Counter()

for labels in events_clean["violation_labels_unique"]:
    if len(labels) >= 2:
        offence_pair_counter.update(
            combinations(sorted(labels), 2)
        )

offence_cooccurrence = pd.DataFrame(
    [
        {
            "violation_1": pair[0],
            "violation_2": pair[1],
            "cooccurrence_count": count
        }
        for pair, count in offence_pair_counter.items()
    ]
).sort_values(
    "cooccurrence_count",
    ascending=False
).reset_index(drop=True)

display(offence_cooccurrence.head(30))

,violation_1,violation_2,cooccurrence_count
0,PARKING IN A MAIN ROAD,WRONG PARKING,15650
1,NO PARKING,PARKING IN A MAIN ROAD,11276
2,NO PARKING,WRONG PARKING,5577
3,DEFECTIVE NUMBER PLATE,WRONG PARKING,4771
4,DEFECTIVE NUMBER PLATE,NO PARKING,3339
5,PARKING ON FOOTPATH,WRONG PARKING,2558
6,NO PARKING,PARKING ON FOOTPATH,1908
7,NO PARKING,PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC,1571
8,DOUBLE PARKING,WRONG PARKING,1400
9,PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC,WRONG PARKING,1325


In [50]:
# Create semantic offence indicators

canonical_text = events_clean[
    "violation_combo_canonical"
]

semantic_patterns = {
    "main_road_flag": r"MAIN ROAD",
    "road_crossing_flag": r"ROAD CROSSING|CROSSING",
    "junction_offence_flag": r"JUNCTION",
    "footpath_flag": r"FOOTPATH",
    "no_parking_flag": r"\bNO PARKING\b",
    "wrong_parking_flag": r"\bWRONG PARKING\b"
}

for feature, pattern in semantic_patterns.items():
    events_clean[feature] = (
        canonical_text
        .str.contains(pattern, regex=True, na=False)
        .astype("int8")
    )

context_columns = [
    "main_road_flag",
    "road_crossing_flag",
    "junction_offence_flag",
    "footpath_flag"
]

events_clean["explicit_obstruction_context_flag"] = (
    events_clean[context_columns]
    .max(axis=1)
    .astype("int8")
)

semantic_flag_summary = pd.DataFrame({
    "feature": list(semantic_patterns.keys())
    + ["explicit_obstruction_context_flag"],
    "event_count": [
        int(events_clean[col].sum())
        for col in list(semantic_patterns.keys())
        + ["explicit_obstruction_context_flag"]
    ]
})

semantic_flag_summary["event_percent"] = (
    semantic_flag_summary["event_count"]
    / len(events_clean)
    * 100
)

display(semantic_flag_summary)

,feature,event_count,event_percent
0,main_road_flag,23943,8.0226
1,road_crossing_flag,1687,0.5653
2,junction_offence_flag,0,0.0000
3,footpath_flag,3757,1.2589
4,no_parking_flag,139048,46.5908
5,wrong_parking_flag,164974,55.2779
6,explicit_obstruction_context_flag,28445,9.5311


In [51]:
# Inspect labels not covered by contextual indicators
context_pattern = re.compile(
    r"MAIN ROAD|ROAD CROSSING|CROSSING|JUNCTION|FOOTPATH"
)

uncovered_atomic_labels = (
    atomic_offence_frequency.loc[
        ~atomic_offence_frequency["violation_label"]
        .str.contains(context_pattern, na=False)
    ]
)

display(uncovered_atomic_labels.head(30))

,violation_label,occurrence_count,unique_event_count,unique_codes,event_percent
0,WRONG PARKING,164974,164974,1,55.2779
1,NO PARKING,139048,139048,1,46.5908
3,DEFECTIVE NUMBER PLATE,7847,7847,1,2.6293
5,PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC,2403,2403,1,0.8052
6,DOUBLE PARKING,2037,2037,1,0.6825
8,REFUSE TO GO FOR HIRE,887,887,1,0.2972
9,PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS,525,525,1,0.1759
10,PARKING OPPOSITE TO ANOTHER PARKED VEHICLE,486,486,1,0.1628
11,USING BLACK FILM/OTHER MATERIALS,248,248,1,0.0831
12,PARKING OTHER THAN BUS STOP,242,242,1,0.0811


In [52]:
# standardize vehicle type text

vehicle_alias_map = {
    "MOTORCYCLE": "MOTOR CYCLE",
    "MOTOR-CYCLE": "MOTOR CYCLE",
    "MOTOR BIKE": "MOTOR CYCLE",
    "AUTO": "PASSENGER AUTO",
    "AUTO RICKSHAW": "PASSENGER AUTO",
    "GOODS VEHICLE": "LORRY/GOODS VEHICLE"
}

events_clean["vehicle_type_clean"] = (
    events_clean["vehicle_type_event"]
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace(vehicle_alias_map)
)

vehicle_type_frequency = (
    events_clean["vehicle_type_clean"]
    .value_counts(dropna=False)
    .rename_axis("vehicle_type_clean")
    .reset_index(name="event_count")
)

vehicle_type_frequency["event_percent"] = (
    vehicle_type_frequency["event_count"]
    / len(events_clean)
    * 100
)

display(vehicle_type_frequency)

,vehicle_type_clean,event_count,event_percent
0,SCOOTER,94856,31.7834
1,CAR,88868,29.7770
2,MOTOR CYCLE,40811,13.6745
3,PASSENGER AUTO,37813,12.6700
4,MAXI-CAB,11372,3.8104
5,LGV,8254,2.7657
6,GOODS AUTO,2934,0.9831
7,MOPED,2199,0.7368
8,PRIVATE BUS,1633,0.5472
9,VAN,1466,0.4912


In [53]:
# create broad vehicle groups
two_wheeler_types = {
    "SCOOTER",
    "MOTOR CYCLE",
    "MOPED"
}

passenger_vehicle_types = {
    "CAR",
    "JEEP"
}

auto_types = {
    "PASSENGER AUTO",
    "GOODS AUTO"
}

light_commercial_types = {
    "MAXI-CAB",
    "LGV",
    "VAN",
    "TEMPO"
}

large_vehicle_types = {
    "HGV",
    "LORRY/GOODS VEHICLE",
    "PRIVATE BUS",
    "BUS (BMTC/KSRTC)"
}
def assign_vehicle_group(vehicle_type):
    if vehicle_type in two_wheeler_types:
        return "TWO_WHEELER"

    if vehicle_type in passenger_vehicle_types:
        return "PASSENGER_VEHICLE"

    if vehicle_type in auto_types:
        return "AUTO_RICKSHAW"

    if vehicle_type in light_commercial_types:
        return "LIGHT_COMMERCIAL"

    if vehicle_type in large_vehicle_types:
        return "LARGE_VEHICLE"

    if vehicle_type == "UNKNOWN VEHICLE TYPE":
        return "UNKNOWN"

    return "OTHER"


events_clean["vehicle_group"] = (
    events_clean["vehicle_type_clean"]
    .map(assign_vehicle_group)
    .astype("category")
)

events_clean["large_vehicle_flag"] = (
    events_clean["vehicle_group"]
    .eq("LARGE_VEHICLE")
    .astype("int8")
)

vehicle_group_summary = (
    events_clean["vehicle_group"]
    .value_counts(dropna=False)
    .rename_axis("vehicle_group")
    .reset_index(name="event_count")
)

vehicle_group_summary["event_percent"] = (
    vehicle_group_summary["event_count"]
    / len(events_clean)
    * 100
)

display(vehicle_group_summary)

,vehicle_group,event_count,event_percent
0,TWO_WHEELER,137866,46.1948
1,PASSENGER_VEHICLE,89781,30.0829
2,AUTO_RICKSHAW,40747,13.6531
3,LIGHT_COMMERCIAL,22458,7.5250
4,LARGE_VEHICLE,5180,1.7357
5,OTHER,2413,0.8085


In [54]:
# inspecting vehicle types assigned to other
other_vehicle_types = (
    events_clean.loc[
        events_clean["vehicle_group"] == "OTHER",
        "vehicle_type_clean"
    ]
    .value_counts()
    .rename_axis("vehicle_type_clean")
    .reset_index(name="event_count")
)

display(other_vehicle_types)

,vehicle_type_clean,event_count
0,OTHERS,895
1,TOURIST BUS,379
2,SCHOOL VEHICLE,378
3,TANKER,260
4,FACTORY BUS,238
5,MINI LORRY,199
6,TRACTOR,64


In [55]:
# validating parsed dataset
assert len(events_clean) == 298445
assert events_clean["id"].is_unique
assert events_clean["vehicle_group"].notna().all()
assert events_clean["violation_combo_canonical"].notna().all()
assert (
    events_clean["violation_count"]
    <= events_clean["violation_count_raw"]
).all()

print("All Response 3 validation checks passed.")
print("Updated shape:", events_clean.shape)

All Response 3 validation checks passed.
Updated shape: (298445, 58)


In [56]:
# saving response outputs
events_parsed_path = (
    OUTPUT_DIR / "events_parsed_stage2.parquet"
)

event_offence_path = (
    OUTPUT_DIR / "event_offence_long.parquet"
)

events_clean.to_parquet(
    events_parsed_path,
    index=False
)

event_offence_long.to_parquet(
    event_offence_path,
    index=False
)

offence_parsing_audit.to_csv(
    OUTPUT_DIR / "offence_parsing_audit.csv",
    index=False
)

atomic_offence_frequency.to_csv(
    OUTPUT_DIR / "atomic_offence_frequency.csv",
    index=False
)

offence_cooccurrence.to_csv(
    OUTPUT_DIR / "offence_cooccurrence.csv",
    index=False
)

vehicle_type_frequency.to_csv(
    OUTPUT_DIR / "vehicle_type_frequency.csv",
    index=False
)

vehicle_group_summary.to_csv(
    OUTPUT_DIR / "vehicle_group_summary.csv",
    index=False
)

column_role_table.to_csv(
    OUTPUT_DIR / "column_roles.csv",
    index=False
)

print("Saved:", events_parsed_path)
print("Saved:", event_offence_path)

Saved: /kaggle/working/parking_intelligence/events_parsed_stage2.parquet
Saved: /kaggle/working/parking_intelligence/event_offence_long.parquet


In [57]:
display(offence_parsing_audit.head())
display(atomic_offence_frequency.head(30))
display(vehicle_type_frequency.head(10))
display(vehicle_group_summary.head())


,metric,value
0,total_events,298445
1,violation_parse_failures,0
2,offence_code_parse_failures,0
3,events_with_zero_labels,0
4,events_with_zero_codes,0


,violation_label,occurrence_count,unique_event_count,unique_codes,event_percent
0,WRONG PARKING,164974,164974,1,55.2779
1,NO PARKING,139048,139048,1,46.5908
2,PARKING IN A MAIN ROAD,23943,23943,1,8.0226
3,DEFECTIVE NUMBER PLATE,7847,7847,1,2.6293
4,PARKING ON FOOTPATH,3757,3757,1,1.2589
5,PARKING NEAR BUSTOP/SCHOOL/HOSPITAL ETC,2403,2403,1,0.8052
6,DOUBLE PARKING,2037,2037,1,0.6825
7,PARKING NEAR ROAD CROSSING,1687,1687,1,0.5653
8,REFUSE TO GO FOR HIRE,887,887,1,0.2972
9,PARKING NEAR TRAFFIC LIGHT OR ZEBRA CROSS,525,525,1,0.1759


,vehicle_type_clean,event_count,event_percent
0,SCOOTER,94856,31.7834
1,CAR,88868,29.7770
2,MOTOR CYCLE,40811,13.6745
3,PASSENGER AUTO,37813,12.6700
4,MAXI-CAB,11372,3.8104
5,LGV,8254,2.7657
6,GOODS AUTO,2934,0.9831
7,MOPED,2199,0.7368
8,PRIVATE BUS,1633,0.5472
9,VAN,1466,0.4912


,vehicle_group,event_count,event_percent
0,TWO_WHEELER,137866,46.1948
1,PASSENGER_VEHICLE,89781,30.0829
2,AUTO_RICKSHAW,40747,13.6531
3,LIGHT_COMMERCIAL,22458,7.5250
4,LARGE_VEHICLE,5180,1.7357


## Diagnose Repeated Captures

In [58]:
# correcting the vehicle groups
large_vehicle_types.update({
    "TOURIST BUS",
    "SCHOOL VEHICLE",
    "FACTORY BUS",
    "TANKER",
    "MINI LORRY",
    "TRACTOR"
})

events_clean["vehicle_group"] = (
    events_clean["vehicle_type_clean"]
    .map(assign_vehicle_group)
    .astype("category")
)

events_clean["large_vehicle_flag"] = (
    events_clean["vehicle_group"]
    .eq("LARGE_VEHICLE")
    .astype("int8")
)

vehicle_group_summary = (
    events_clean["vehicle_group"]
    .value_counts(dropna=False)
    .rename_axis("vehicle_group")
    .reset_index(name="event_count")
)

vehicle_group_summary["event_percent"] = (
    vehicle_group_summary["event_count"]
    / len(events_clean) * 100
)

display(vehicle_group_summary)

,vehicle_group,event_count,event_percent
0,TWO_WHEELER,137866,46.1948
1,PASSENGER_VEHICLE,89781,30.0829
2,AUTO_RICKSHAW,40747,13.6531
3,LIGHT_COMMERCIAL,22458,7.5250
4,LARGE_VEHICLE,6698,2.2443
5,OTHER,895,0.2999


In [59]:
# Sorts each vehicle’s records chronologically and compares every record with its previous observation.
 #It calculates time gap, geographic distance, device agreement, creator agreement and offence agreement.
ordered = events_clean.sort_values(
    ["vehicle_token", "created_at_utc", "source_row"]
).copy()

grouped = ordered.groupby("vehicle_token", sort=False)

for col in [
    "created_at_utc",
    "latitude",
    "longitude",
    "device_id",
    "created_by_id",
    "violation_combo_canonical"
]:
    ordered[f"previous_{col}"] = grouped[col].shift(1)

ordered["time_gap_seconds"] = (
    ordered["created_at_utc"]
    - ordered["previous_created_at_utc"]
).dt.total_seconds()

lat1 = np.radians(ordered["previous_latitude"])
lat2 = np.radians(ordered["latitude"])
dlat = lat2 - lat1
dlon = np.radians(
    ordered["longitude"]
    - ordered["previous_longitude"]
)

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1)
    * np.cos(lat2)
    * np.sin(dlon / 2) ** 2
)

ordered["distance_metres"] = (
    2 * 6_371_000
    * np.arctan2(
        np.sqrt(a.clip(0, 1)),
        np.sqrt((1 - a).clip(0, 1))
    )
)

ordered["same_device"] = (
    ordered["device_id"]
    == ordered["previous_device_id"]
)

ordered["same_creator"] = (
    ordered["created_by_id"]
    == ordered["previous_created_by_id"]
)

ordered["same_offence_combo"] = (
    ordered["violation_combo_canonical"]
    == ordered["previous_violation_combo_canonical"]
)

In [60]:
# comparing possible duplicate threshold 
threshold_results = []

for seconds, metres in [
    (60, 10),
    (120, 20),
    (300, 30),
    (600, 50)
]:
    mask = (
        ordered["time_gap_seconds"].between(0, seconds)
        & ordered["distance_metres"].le(metres)
    )

    count = int(mask.sum())

    threshold_results.append({
        "maximum_time_seconds": seconds,
        "maximum_distance_metres": metres,
        "candidate_pairs": count,
        "same_device_percent": (
            ordered.loc[mask, "same_device"].mean() * 100
            if count else 0
        ),
        "same_creator_percent": (
            ordered.loc[mask, "same_creator"].mean() * 100
            if count else 0
        ),
        "same_offence_percent": (
            ordered.loc[mask, "same_offence_combo"].mean() * 100
            if count else 0
        ),
        "same_device_creator_offence_percent": (
            (
                ordered.loc[mask, "same_device"]
                & ordered.loc[mask, "same_creator"]
                & ordered.loc[mask, "same_offence_combo"]
            ).mean() * 100
            if count else 0
        )
    })

threshold_comparison = pd.DataFrame(threshold_results)

display(threshold_comparison)

,maximum_time_seconds,maximum_distance_metres,candidate_pairs,same_device_percent,same_creator_percent,same_offence_percent,same_device_creator_offence_percent
0,60,10,5729,98.2894,98.2719,98.1498,97.1374
1,120,20,6040,96.1093,96.1093,97.1854,94.6192
2,300,30,6549,93.1898,93.2661,95.4650,91.0215
3,600,50,7315,89.1046,89.3233,93.2741,86.2064


In [61]:
# create conservative incident groups
ordered["duplicate_link_to_previous"] = (
    ordered["time_gap_seconds"].between(0, 120)
    & ordered["distance_metres"].le(20)
    & ordered["same_device"]
    & ordered["same_creator"]
    & ordered["same_offence_combo"]
)

ordered["new_incident"] = (
    ~ordered["duplicate_link_to_previous"]
)

ordered["incident_sequence"] = (
    ordered.groupby("vehicle_token", sort=False)["new_incident"]
    .cumsum()
)

ordered["incident_group_id"] = (
    ordered["vehicle_token"].astype(str)
    + "_"
    + ordered["incident_sequence"].astype(str)
)

ordered["capture_count"] = (
    ordered.groupby("incident_group_id")["source_row"]
    .transform("size")
)

incident_links = ordered[
    [
        "source_row",
        "incident_group_id",
        "capture_count",
        "duplicate_link_to_previous",
        "time_gap_seconds",
        "distance_metres"
    ]
]

events_with_incidents = events_clean.merge(
    incident_links,
    on="source_row",
    how="left",
    validate="one_to_one"
)

incident_summary = pd.DataFrame({
    "metric": [
        "total_event_records",
        "unique_incident_groups",
        "multi_capture_incident_groups",
        "records_inside_multi_capture_groups",
        "records_removed_if_collapsed",
        "maximum_captures_in_one_group"
    ],
    "value": [
        len(events_with_incidents),
        events_with_incidents["incident_group_id"].nunique(),
        events_with_incidents.loc[
            events_with_incidents["capture_count"] > 1,
            "incident_group_id"
        ].nunique(),
        int(
            (
                events_with_incidents["capture_count"] > 1
            ).sum()
        ),
        len(events_with_incidents)
        - events_with_incidents["incident_group_id"].nunique(),
        int(events_with_incidents["capture_count"].max())
    ]
})

display(incident_summary)

,metric,value
0,total_event_records,298445
1,unique_incident_groups,292730
2,multi_capture_incident_groups,3609
3,records_inside_multi_capture_groups,9324
4,records_removed_if_collapsed,5715
5,maximum_captures_in_one_group,15


In [62]:
# inspect and save candidate groups
candidate_incidents = (
    events_with_incidents.loc[
        events_with_incidents["capture_count"] > 1,
        [
            "incident_group_id",
            "capture_count",
            "id",
            "vehicle_token",
            "created_at_ist",
            "latitude",
            "longitude",
            "device_id",
            "created_by_id",
            "violation_combo_canonical",
            "time_gap_seconds",
            "distance_metres"
        ]
    ]
    .sort_values(
        [
            "capture_count",
            "incident_group_id",
            "created_at_ist"
        ],
        ascending=[False, True, True]
    )
)

display(candidate_incidents.head(30))

events_with_incidents.to_parquet(
    OUTPUT_DIR / "events_with_incident_groups.parquet",
    index=False
)

threshold_comparison.to_csv(
    OUTPUT_DIR / "duplicate_threshold_comparison.csv",
    index=False
)

incident_summary.to_csv(
    OUTPUT_DIR / "incident_group_summary.csv",
    index=False
)

,incident_group_id,capture_count,id,vehicle_token,created_at_ist,latitude,longitude,device_id,created_by_id,violation_combo_canonical,time_gap_seconds,distance_metres
28897,574a9b9d67690e9724e1_1,15,FKID028897,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,NaN,NaN
28911,574a9b9d67690e9724e1_1,15,FKID028911,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,0.0000,0.0000
28912,574a9b9d67690e9724e1_1,15,FKID028912,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,0.0000,0.0000
28942,574a9b9d67690e9724e1_1,15,FKID028942,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,0.0000,0.0000
104609,574a9b9d67690e9724e1_1,15,FKID104610,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,0.0000,0.0000
104620,574a9b9d67690e9724e1_1,15,FKID104621,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,0.0000,0.0000
104625,574a9b9d67690e9724e1_1,15,FKID104626,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,0.0000,0.0000
104626,574a9b9d67690e9724e1_1,15,FKID104627,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,0.0000,0.0000
104630,574a9b9d67690e9724e1_1,15,FKID104631,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,0.0000,0.0000
104631,574a9b9d67690e9724e1_1,15,FKID104632,574a9b9d67690e9724e1,2024-03-13 06:28:46+05:30,12.8763,77.5965,FKDEV00628,FKUSR00609,WRONG PARKING,0.0000,0.0000


## Finalize Incident Deduplication

In [63]:
# validate complete duplicate groups
duplicate_work = events_with_incidents.sort_values(
    ["incident_group_id", "created_at_utc", "source_row"]
).copy()

duplicate_work["first_latitude"] = (
    duplicate_work.groupby("incident_group_id")["latitude"]
    .transform("first")
)

duplicate_work["first_longitude"] = (
    duplicate_work.groupby("incident_group_id")["longitude"]
    .transform("first")
)

lat1 = np.radians(duplicate_work["first_latitude"])
lat2 = np.radians(duplicate_work["latitude"])
dlat = lat2 - lat1
dlon = np.radians(
    duplicate_work["longitude"]
    - duplicate_work["first_longitude"]
)

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

duplicate_work["distance_from_first_metres"] = (
    2 * 6_371_000
    * np.arctan2(
        np.sqrt(a.clip(0, 1)),
        np.sqrt((1 - a).clip(0, 1))
    )
)

group_audit = (
    duplicate_work.groupby("incident_group_id")
    .agg(
        capture_count=("source_row", "size"),
        group_start_utc=("created_at_utc", "min"),
        group_end_utc=("created_at_utc", "max"),
        max_distance_from_first_metres=(
            "distance_from_first_metres",
            "max"
        ),
        unique_devices=("device_id", "nunique"),
        unique_creators=("created_by_id", "nunique"),
        unique_offence_combinations=(
            "violation_combo_canonical",
            "nunique"
        )
    )
    .reset_index()
)

group_audit["group_duration_seconds"] = (
    group_audit["group_end_utc"]
    - group_audit["group_start_utc"]
).dt.total_seconds()

group_audit["confirmed_duplicate_group"] = (
    group_audit["capture_count"].gt(1)
    & group_audit["group_duration_seconds"].le(120)
    & group_audit["max_distance_from_first_metres"].le(20)
    & group_audit["unique_devices"].eq(1)
    & group_audit["unique_creators"].eq(1)
    & group_audit["unique_offence_combinations"].eq(1)
)

confirmation_summary = pd.DataFrame({
    "metric": [
        "candidate_multi_capture_groups",
        "confirmed_duplicate_groups",
        "unconfirmed_candidate_groups",
        "records_in_confirmed_groups",
        "records_that_will_be_collapsed"
    ],
    "value": [
        int((group_audit["capture_count"] > 1).sum()),
        int(group_audit["confirmed_duplicate_group"].sum()),
        int(
            (
                (group_audit["capture_count"] > 1)
                & ~group_audit["confirmed_duplicate_group"]
            ).sum()
        ),
        int(
            group_audit.loc[
                group_audit["confirmed_duplicate_group"],
                "capture_count"
            ].sum()
        ),
        int(
            (
                group_audit.loc[
                    group_audit["confirmed_duplicate_group"],
                    "capture_count"
                ] - 1
            ).sum()
        )
    ]
})

display(confirmation_summary)

,metric,value
0,candidate_multi_capture_groups,3609
1,confirmed_duplicate_groups,3609
2,unconfirmed_candidate_groups,0
3,records_in_confirmed_groups,9324
4,records_that_will_be_collapsed,5715


In [64]:
# collapse only confirmed duplicate groups
confirmed_group_ids = set(
    group_audit.loc[
        group_audit["confirmed_duplicate_group"],
        "incident_group_id"
    ]
)

duplicate_work["final_incident_id"] = np.where(
    duplicate_work["incident_group_id"].isin(confirmed_group_ids),
    duplicate_work["incident_group_id"],
    "ROW_" + duplicate_work["source_row"].astype(str)
)

base_columns = [
    col for col in events_clean.columns
    if col in duplicate_work.columns
]

representative_events = (
    duplicate_work.sort_values(
        ["final_incident_id", "created_at_utc", "source_row"]
    )
    .drop_duplicates("final_incident_id", keep="first")
    [base_columns + ["final_incident_id"]]
    .copy()
)

incident_aggregates = (
    duplicate_work.groupby("final_incident_id")
    .agg(
        duplicate_capture_count=("source_row", "size"),
        incident_start_utc=("created_at_utc", "min"),
        incident_end_utc=("created_at_utc", "max"),
        incident_latitude=("latitude", "median"),
        incident_longitude=("longitude", "median"),
        validation_records_audit=(
            "validation_status",
            lambda x: int(x.notna().sum())
        ),
        validation_statuses_audit=(
            "validation_status",
            lambda x: " | ".join(
                sorted(x.dropna().astype(str).unique())
            )
        ),
        scita_sent_any_audit=(
            "data_sent_to_scita",
            lambda x: bool(x.fillna(False).any())
        )
    )
    .reset_index()
)

incidents_clean = representative_events.merge(
    incident_aggregates,
    on="final_incident_id",
    how="left",
    validate="one_to_one"
)

incidents_clean["incident_duration_seconds"] = (
    incidents_clean["incident_end_utc"]
    - incidents_clean["incident_start_utc"]
).dt.total_seconds()

incidents_clean["records_collapsed"] = (
    incidents_clean["duplicate_capture_count"] - 1
)

incidents_clean["was_multi_capture_incident"] = (
    incidents_clean["duplicate_capture_count"] > 1
)

# Use median coordinates for confirmed repeated captures.
incidents_clean["latitude"] = incidents_clean["incident_latitude"]
incidents_clean["longitude"] = incidents_clean["incident_longitude"]

In [65]:
# validate final incident dataset
deduplication_summary = pd.DataFrame({
    "metric": [
        "records_before_deduplication",
        "incidents_after_deduplication",
        "records_collapsed",
        "percent_records_collapsed",
        "multi_capture_incidents",
        "maximum_duplicate_capture_count"
    ],
    "value": [
        len(events_clean),
        len(incidents_clean),
        len(events_clean) - len(incidents_clean),
        (
            (len(events_clean) - len(incidents_clean))
            / len(events_clean) * 100
        ),
        int(incidents_clean["was_multi_capture_incident"].sum()),
        int(incidents_clean["duplicate_capture_count"].max())
    ]
})

assert incidents_clean["final_incident_id"].is_unique
assert incidents_clean["id"].is_unique
assert len(incidents_clean) <= len(events_clean)

assert (
    incidents_clean["records_collapsed"].sum()
    == len(events_clean) - len(incidents_clean)
)

capture_count_distribution = (
    incidents_clean["duplicate_capture_count"]
    .value_counts()
    .sort_index()
    .rename_axis("duplicate_capture_count")
    .reset_index(name="incident_count")
)

display(deduplication_summary)
display(capture_count_distribution)

,metric,value
0,records_before_deduplication,"298,445.0000"
1,incidents_after_deduplication,"292,730.0000"
2,records_collapsed,"5,715.0000"
3,percent_records_collapsed,1.9149
4,multi_capture_incidents,"3,609.0000"
5,maximum_duplicate_capture_count,15.0000


,duplicate_capture_count,incident_count
0,1,289121
1,2,2654
2,3,498
3,4,213
4,5,73
5,6,57
6,7,43
7,8,32
8,9,19
9,10,6


In [66]:
# save the incident level dataset
events_incident = incidents_clean.copy()

events_incident.to_parquet(
    OUTPUT_DIR / "events_incident_level.parquet",
    index=False
)

group_audit.to_csv(
    OUTPUT_DIR / "duplicate_group_audit.csv",
    index=False
)

confirmation_summary.to_csv(
    OUTPUT_DIR / "duplicate_confirmation_summary.csv",
    index=False
)

deduplication_summary.to_csv(
    OUTPUT_DIR / "deduplication_summary.csv",
    index=False
)

print("Final incident-level shape:", events_incident.shape)
print("Saved incident-level dataset.")

Final incident-level shape: (292730, 70)
Saved incident-level dataset.


In [67]:
events_incident.shape

(292730, 70)